In [1]:
import os
import re
import zipfile

import folium
import pandas as pd
import polars as pl
import polars_h3 as plh3
import polars_xdt as xdt

In [2]:
# parameters
city_name = "cologne"
mode = "nextbike"
bike_file = f"{mode}_availability_{city_name.lower()}.csv"
bike_zip_file = f"../data/sharing_locations_raw/{bike_file}.zip"
geometa_path = f"../data/stateful_variables/{city_name.lower()}_geometa.pkl"
sharing_positions_path = (
    f"../data/sharing_locations_clustered/{city_name.lower()}_{mode}"
)
# city_center = [52.5170365, 13.3888599]
city_center = [50.938361, 6.959974]

In [3]:
with zipfile.ZipFile(bike_zip_file).open(bike_file) as f:
    availabilities = pl.read_csv(f)

availabilities = availabilities.rename(
    lambda name: name.replace(f"{mode}_availability_", "")
)
availabilities = availabilities.with_columns(
    pl.col("geometry").alias("geometry_str"),
    pl.col("geometry")
    .str.replace(r"[a-zA-Z\(]+", "")
    .str.replace(r"\)", "")
    .str.split(by=" ")
    .list.to_struct(fields=["lon", "lat"], n_field_strategy="max_width"),
).unnest("geometry")
availabilities = availabilities.with_columns(
    pl.col("lon").cast(pl.Float64),
    pl.col("lat").cast(pl.Float64),
    availabilities["valid_from"].str.to_datetime(),
    availabilities["valid_till"].str.to_datetime(),
)
availabilities = availabilities.with_columns(
    plh3.latlng_to_cell("lat", "lon", resolution=8, return_dtype=pl.Utf8).alias(
        "h3_cell"
    ),
    xdt.ceil("valid_from", "1h"),
    pl.col("valid_till").dt.truncate("1h"),
)

In [4]:
print(
    availabilities["valid_from"].min(),
    availabilities["valid_till"].max(),
)

2022-01-15 01:00:00 2024-04-30 23:00:00


## Spatial and Temporal Discretization 

In [5]:
availabilities = availabilities.filter(pl.col("valid_from") <= pl.col("valid_till"))

In [ ]:
availabilities = availabilities.with_columns(
    pl.struct("valid_from", "valid_till")
    .map_elements(
        lambda row: pl.datetime_range(
            row["valid_from"], row["valid_till"], "1h", closed="both", eager=True
        ),
        return_dtype=pl.List(pl.Datetime),
    )
    .alias("interval")
)

In [ ]:
availabilities.head()

In [ ]:
n_bikes_per_hex_per_time = pd.DataFrame(n_bikes_per_hex_per_time)
n_bikes_per_hex_per_time.index.name = "time"

In [ ]:
n_bikes_per_hex_per_time.head()

In [ ]:
n_bikes_per_hex_per_time.to_csv(
    f"../data/sharing_locations_raw/{mode}_availability_{
        city_name.lower()}_bucketed.csv.zip"
)

In [ ]:
n_bikes_per_hex_per_time = pd.read_csv(
    f"../data/sharing_locations_raw/{mode}_availability_{
        city_name.lower()}_bucketed.csv.zip",
    index_col=0,
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_bikes = scaler.fit_transform(n_bikes_per_hex_per_time)

In [ ]:
from sklearn.metrics import calinski_harabasz_score
from sklearn_extra.cluster import KMedoids

stats = []
for k in range(2, 15):
    model = KMedoids(n_clusters=k, random_state=4711)
    pred_ = model.fit_predict(scaled_bikes)
    stats.append(
        {
            "k": k,
            "calinski_harabasz_score": calinski_harabasz_score(
                n_bikes_per_hex_per_time, pred_
            ),
        }
    )

In [ ]:
pd.DataFrame(stats).plot(x="k", y="calinski_harabasz_score")
# plt.savefig(f"../figures/sharing_clustering/{city_name}_{mode}_calinski_harabasz.png")

In [ ]:
# Number of clusters you want
n_clusters = 5

model = KMedoids(n_clusters=n_clusters, random_state=4711)
model.fit(scaled_bikes)

In [ ]:
pred = model.predict(scaled_bikes)

In [ ]:
pred

In [ ]:
n_bikes_per_hex_per_time["pred"] = pred

In [ ]:
visual = n_bikes_per_hex_per_time[
    n_bikes_per_hex_per_time.columns[n_bikes_per_hex_per_time.nunique() != 1]
]

In [ ]:
import seaborn as sns

sns.pairplot(visual, vars=visual.columns[:-1], hue="pred")

In [ ]:
medoids = n_bikes_per_hex_per_time.iloc[model.medoid_indices_]

In [ ]:
medoids

In [ ]:
from mcr_py.package.geometa import GeoMeta

geo_meta = GeoMeta.load(geometa_path)

In [ ]:
m = folium.Map(location=city_center, zoom_start=12)
geo_meta.add_to_folium_map(m)
h3.plot_h3_cells_on_folium(medoids.iloc[0].to_dict(), m, popup_callback=lambda x, y: x)
m

In [ ]:
def get_locations_at_time():
    pass


time = medoids.index[3]
locations_at_time = get_locations_at_time(availabilities, time)
time

In [ ]:
medoids.index

In [ ]:
# center in cologne
m = folium.Map(location=city_center, zoom_start=13)

colors = ["red", "blue", "green", "yellow", "purple", "orange", "brown"]

for i, time in enumerate(medoids.index):
    locations_at_time = get_locations_at_time(availabilities, time)
    for point in locations_at_time[["lat", "lon"]].values:
        folium.CircleMarker(
            location=point, radius=2, color=colors[i], fill=True, fill_color="#000000"
        ).add_to(m)


m

In [ ]:
def derive_filename(s) -> str:
    s = re.sub(r"[^a-zA-Z0-9\-_.]", "_", str(s))
    s = s.replace(" ", "_")
    s = re.sub(r"_+", "_", s)
    return s

In [ ]:
os.makedirs(sharing_positions_path, exist_ok=True)
for time in medoids.index:
    locations_at_time = get_locations_at_time(availabilities, time)
    filename = derive_filename(time) + ".csv"
    file_path = os.path.join(sharing_positions_path, filename)
    locations_at_time[["lat", "lon"]].to_csv(file_path, index=False)